## **DS522-FHIR Data Obfuscation for Population-Level Analytics**
## **DS522: Data Acquisition and Analytics - USADL - Spring 2025-2026, PyForce**
>   **School of Technology and Computing**<br>
    **City University of Seattle** <br>
    *Seattle, WA 98121* <br>

`Makenzy Abass Gumah, `<br> 
`gumahmakenzyabass@cityuniversity.edu`

`Jacqueline Biira, `<br>
`biirajacqueline@cityuniversity.edu`

`Dawit Hailu, MSAI`<br>
`hailudawith@cityuniversity.edu`

### **Bulk Import FHIR data:**

#### *Library Imports:*

In [1]:
from bulk_import import FHIRBulkLoader # import loader class from bulk_import.py
import pandas as pd
import requests

In [ ]:

from bulk_import import FHIRBulkLoader

FHIR_SERVER = "http://107.22.53.0:8080/fhir"
ZIP_FILE = "synthea_sample_data_fhir_r4_nov2021.zip" #initial data loaded Ref: https://mitre.box.com/shared/static/ylzmiichhvtw1igr4ck6q32i5b333nqs.zip
COVID_ZIP_FILE = "10k_synthea_covid19_csv.zip" #COVID-19 data loaded Ref: https://mitre.box.com/shared/static/9iglv8kbs1pfi7z8phjl9sbpjk08spze.zip

loader = FHIRBulkLoader(base_url=FHIR_SERVER, max_workers=2)
# loader.load_from_zip(ZIP_FILE)
loader.load_from_zip(COVID_ZIP_FILE, file_type='csv')

### **Test API Access:**

In [4]:
def get_patient_data(base_url):
    # Search for all Patients
    response = requests.get(f"{base_url}/Patient?_summary=true", timeout=30)
    if response.status_code == 200:
        bundle = response.json()
        # Extract specific fields for analysis
        patients = []
        for entry in bundle.get('entry', []):
            resource = entry['resource']
            patients.append({
                'id': resource.get('id'),
                'gender': resource.get('gender'),
                'birthDate': resource.get('birthDate')
            })
        return pd.DataFrame(patients)
    return pd.DataFrame()

In [6]:
# get patient cout
response = requests.get(f"{FHIR_SERVER}/Patient?_summary=count", timeout=30)
if response.status_code == 200:
    count = response.json().get('total', 0)
    print(f"Total Patients: {count}")
else:
    print(f"Failed to retrieve patient count: {response.status_code}")
# view the first few rows of the patient data
df = get_patient_data(FHIR_SERVER)
print(df.head())

Total Patients: 849
       id  gender   birthDate
0  253927  female  1957-02-12
1  258835  female  1975-08-05
2  259649    male  1976-04-15
3  263685  female  2007-08-02
4  265003    male  1981-03-10


In [4]:

import psycopg2
import boto3

password = "dbpass26"

conn = None
try:
    conn = psycopg2.connect(
        host='hapi-fhir-db.cwt602ay6kje.us-east-1.rds.amazonaws.com',
        port=5432,
        database='postgres',
        user='postgres',
        password=password,
        sslmode='verify-full',
    sslrootcert='./global-bundle.pem'
    )
    cur = conn.cursor()
    cur.execute('SELECT version();')
    print(cur.fetchone()[0])
    cur.close()
except Exception as e:
    print(f"Database error: {e}")
    raise
finally:
    if conn:
        conn.close()

Database error: connection to server at "hapi-fhir-db.cwt602ay6kje.us-east-1.rds.amazonaws.com" (54.204.238.124), port 5432 failed: Connection timed out (0x0000274C/10060)
	Is the server running on that host and accepting TCP/IP connections?



OperationalError: connection to server at "hapi-fhir-db.cwt602ay6kje.us-east-1.rds.amazonaws.com" (54.204.238.124), port 5432 failed: Connection timed out (0x0000274C/10060)
	Is the server running on that host and accepting TCP/IP connections?
